<a href="https://colab.research.google.com/github/raju-133/india-house-price-predictor/blob/main/house_price_predictor_updated_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
from google.colab import files
import io

uploaded = files.upload()
df = pd.read_csv(io.BytesIO(next(iter(uploaded.values()))))

display(df.head())
print(df.shape)

Saving india_house_price_synthetic_15000.csv to india_house_price_synthetic_15000.csv


,Location,Bedrooms,Land_Size_sqft,Price_INR
0,Hyderabad,3,1415,6720000
1,Vijayawada,2,558,1790000
2,Kakinada,3,1598,3670000
3,Kochi,3,941,5570000
4,Hyderabad,5,3316,24230000


(15000, 4)


In [ ]:
X = df[["Location", "Bedrooms", "Land_Size_sqft"]]
y = df["Price_INR"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

In [ ]:
preprocessor = ColumnTransformer([
    ("location", OneHotEncoder(handle_unknown="ignore"),
     ["Location"]),
    ("numbers", SimpleImputer(strategy="median"),
     ["Bedrooms", "Land_Size_sqft"])
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1,
        min_samples_leaf=2
    ))
])

In [ ]:
model.fit(X_train, y_train)

predictions = model.predict(X_test)

print("MAE:", mean_absolute_error(y_test, predictions))
print("RMSE:", np.sqrt(mean_squared_error(y_test, predictions)))
print("R²:", r2_score(y_test, predictions))

MAE: 1453130.1694425398
RMSE: 2484717.597529297
R²: 0.8418506777024933


In [ ]:


location = input("Enter location (e.g., Vijayawada): ").strip()
bedrooms = int(input("Enter number of bedrooms: "))
land_size = float(input("Enter land size in square feet: "))

# Prepare the input for the trained model
new_house = pd.DataFrame([{
    "Location": location,
    "Bedrooms": bedrooms,
    "Land_Size_sqft": land_size
}])

# Predict the price
predicted_price = model.predict(new_house)[0]

print("\n" + "=" * 35)
print(f"📍 Location: {location}")
print(f"🛏️ Bedrooms: {bedrooms}")
print(f"📐 Land size: {land_size:,.0f} sq ft")
print(f"💰 Predicted House Price: ₹{predicted_price:,.0f}")
print("=" * 35)


Enter location (e.g., Vijayawada): vijayawada
Enter number of bedrooms: 2
Enter land size in square feet: 550

📍 Location: vijayawada
🛏️ Bedrooms: 2
📐 Land size: 550 sq ft
💰 Predicted House Price: ₹1,577,295


In [ ]:
joblib.dump(model, "india_house_price_model.joblib")

['india_house_price_model.joblib']